In [3]:
!pip install ta
!pip install yfinance


  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=467f7541a6a64ddaae5683e8eb5415d7584bef2e58522d5af298851262b25a6e
  Stored in directory: /root/.cache/pip/wheels/5f/67/4f/8a9f252836e053e532c6587a3230bc72a4deb16b03a829610b
Successfully built ta


In [1]:
# scalar matrix parallel lstm


import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from torch.utils.checkpoint import checkpoint

class StructuredStateSpace(nn.Module):
    def __init__(self, d_state, d_model, discretization='bilinear', dt_min=0.001, dt_max=0.1):
        super().__init__()
        self.d_state = d_state
        self.d_model = d_model
        self.discretization = discretization
        
        self.log_lambda_real = nn.Parameter(torch.linspace(math.log(dt_min), math.log(dt_max), d_state))
        self.log_b = nn.Parameter(torch.randn(d_state, 1).uniform_(-0.5, 0.5))  # Changed to d_state x 1
        self.c = nn.Parameter(torch.randn(1, d_state) / math.sqrt(d_state))     # Changed to 1 x d_state
        self.log_d = nn.Parameter(torch.zeros(1))                               # Changed to scalar
        
        # Fix: Use a single step size parameter now
        self.log_step = nn.Parameter(torch.tensor(math.log(dt_min)))
        
        self.volatility_gate = nn.Parameter(torch.ones(d_state))
        self.alpha = nn.Parameter(torch.tensor(0.7))
        self.layer_norm = nn.LayerNorm(d_state)

    def forward(self, x, volatility_scale=None):
        batch, seq_len, _ = x.shape
        lambda_real = -torch.exp(self.log_lambda_real)  # [d_state]
        b = torch.exp(self.log_b)                       # [d_state, 1]
        c = self.c                                      # [1, d_state]
        d = torch.exp(self.log_d)                       # [1]
        step = torch.exp(self.log_step)                 # Scalar
        
        # Simplify the discretization calculation
        if self.discretization == 'zoh':
            a_discrete = torch.exp(lambda_real * step)                      # [d_state]
            b_discrete = (a_discrete - 1.0) / lambda_real * b.squeeze(-1)   # [d_state]
        elif self.discretization == 'bilinear':
            a_discrete = (2.0 + step * lambda_real) / (2.0 - step * lambda_real)  # [d_state]
            b_discrete = step * (torch.ones_like(a_discrete) + a_discrete) * b.squeeze(-1) / 2.0  # [d_state]
        
        # Initialize the hidden state
        h = torch.zeros(batch, self.d_state, device=x.device)  # [batch, d_state]
        outputs = []
        
        # Simplify the recurrent loop
        for t in range(seq_len):
            # State update: h = A*h + B*x
            h = a_discrete.unsqueeze(0) * h + b_discrete.unsqueeze(0) * x[:, t, :].mean(dim=1, keepdim=True)
            
            # Apply layer normalization
            h = self.layer_norm(h)
            
            # Apply volatility scaling if provided
            if volatility_scale is not None:
                vol_influence = torch.sigmoid(self.volatility_gate) * volatility_scale[:, t]
                h = h * (1.0 / (1.0 + vol_influence))
            
            # Output calculation: y = C*h + D*x
            y = torch.matmul(h.unsqueeze(1), c.transpose(0, 1)).squeeze(1) + d * x[:, t, :]
            outputs.append(y)
        
        y = torch.stack(outputs, dim=1)
        alpha = torch.sigmoid(self.alpha)
        
        # Residual connection
        return alpha * x + (1 - alpha) * y


class ParallelSMLSTMCell(nn.Module):
    """
    Enhanced mLSTM with structured state-space memory (s_mLSTM) for financial time series.
    """
    def __init__(self, input_size, hidden_size, matrix_size=4, stabilizer_threshold=1.0):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.matrix_size = matrix_size
        self.d = hidden_size  # State dimension
        self.stabilizer_threshold = stabilizer_threshold
        
        # Initialize memory matrix and state vectors
        self.C_init = nn.Parameter(torch.zeros(1, hidden_size, hidden_size))
        self.n_init = nn.Parameter(torch.zeros(1, hidden_size))
        self.m_init = nn.Parameter(torch.zeros(1, 1))
        
        # Input projections
        self.W_q = nn.Linear(input_size, hidden_size)
        self.W_k = nn.Linear(input_size, hidden_size)
        self.W_v = nn.Linear(input_size, hidden_size)
        self.W_i = nn.Linear(input_size, 1)
        self.W_f = nn.Linear(input_size, 1)
        self.W_o = nn.Linear(input_size, hidden_size)
        
        # Structured state-space memory model for each matrix dimension
        self.ssm = StructuredStateSpace(
            d_state=hidden_size,       # Reduced from hidden_size*2 to match properly
            d_model=hidden_size,                 # Simplified to scalar model dimension
            discretization='bilinear',
            dt_min=0.001,
            dt_max=0.1
        )
        
        # Financial regime detection network
        # Detects market regimes like trending, mean-reverting, high-volatility, etc.
        self.regime_detector = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, 4),  # 4 typical market regimes
            nn.Softmax(dim=-1)
        )
        
        # Layer normalization for numerical stability
        self.layer_norm_input = nn.LayerNorm(input_size)
        self.layer_norm_memory = nn.LayerNorm(hidden_size)
        
        # Initialize with financial-specific biases
        # Slightly bias forget gate toward remembering (financial time series have long-term dependencies)
        self.W_f.bias.data.fill_(1.0)
        # Initialize input gate slightly lower to be more selective about new information
        self.W_i.bias.data.fill_(0.0)

    def detect_volatility(self, x):
        """Extract volatility signal from input features for adaptive processing."""
        # Simple volatility estimation: std of last few time steps in sequence
        # Assumes input contains price information
        batch_size, seq_len, _ = x.shape
        
        # Use the last 20% of sequence for volatility estimation (rolling window)
        window_size = max(1, int(seq_len * 0.2))
        if seq_len <= 1:
            return torch.ones(batch_size, seq_len, 1, device=x.device)
            
        # Compute rolling volatility (std of price changes)
        # For simplicity, we use the standard deviation across feature dimensions
        # In practice, extract specific volatility features or financial indicators
        volatility = torch.stack([
            torch.std(x[:, i-window_size:i], dim=1) 
            for i in range(window_size, seq_len+1)
        ], dim=1)
        
        # Pad beginning
        padding = torch.ones(batch_size, window_size-1, x.size(2), device=x.device) * \
                  volatility[:, 0:1, :]
        volatility = torch.cat([padding, volatility], dim=1)
        
        # Extract scalar volatility (mean across features)
        scalar_vol = volatility.mean(dim=2, keepdim=True)
        
        # Normalize and add a small constant for stability
        scalar_vol = scalar_vol / (torch.mean(scalar_vol, dim=1, keepdim=True) + 1e-6)
        
        return scalar_vol
    
    def stabilize_gates(self, i_tilde, f_tilde, m_prev):
        batch_size, seq_len = i_tilde.shape[0], i_tilde.shape[1]
        
        # Ensure m_prev has the right shape for broadcasting
        if m_prev.dim() == 1:
            m_prev = m_prev.unsqueeze(1)  # [batch, 1]
        
        # Reshape m_prev for proper broadcasting
        m_prev_exp = m_prev.unsqueeze(1).expand(batch_size, seq_len, 1).clamp(-100, 100)
        
        # Compute max term for LogSumExp stability
        m_t = torch.maximum(f_tilde, m_prev_exp + f_tilde)
        
        # Stable computation with clipping for financial data extremes
        i = torch.exp(torch.clamp(i_tilde - m_t, -15.0, 15.0))
        f = torch.exp(torch.clamp(f_tilde + m_prev_exp - m_t, -15.0, 15.0))
        
        return i, f, m_t
    def store_key_value(self, key, value, i):
        """
        Compute key-value storage via outer product with memory-efficient implementation.
        """
        batch_size, seq_len, d = key.shape
        
        # Scale keys and values to prevent extremely large outer products
        key_norm = F.normalize(key, dim=2) * math.sqrt(self.d)
        value_norm = value / (torch.norm(value, dim=2, keepdim=True) + 1e-6) * math.sqrt(self.d)
        
        # Ensure same sequence length for all inputs
        min_seq = min(key_norm.shape[1], value_norm.shape[1], i.shape[1])
        key_norm = key_norm[:, :min_seq]
        value_norm = value_norm[:, :min_seq]
        i = i[:, :min_seq]
        
        # Memory-efficient implementation: process in chunks
        chunk_size = min(32, min_seq)  # Process 32 sequence steps at a time
        num_chunks = (min_seq + chunk_size - 1) // chunk_size
        
        result_chunks = []
        for chunk_idx in range(num_chunks):
            start_idx = chunk_idx * chunk_size
            end_idx = min(start_idx + chunk_size, min_seq)
            
            # Process this chunk
            chunk_key = key_norm[:, start_idx:end_idx]
            chunk_value = value_norm[:, start_idx:end_idx]
            chunk_i = i[:, start_idx:end_idx]
            
            # Compute outer product for this chunk only
            outer_product = torch.bmm(
                chunk_value.reshape(batch_size * (end_idx - start_idx), d, 1),
                chunk_key.reshape(batch_size * (end_idx - start_idx), 1, d)
            ).reshape(batch_size, end_idx - start_idx, d, d)
            
            # Apply input gate with gradient clipping for stability
            i_clamped = torch.clamp(chunk_i, 0.0, 1.0)
            result_chunks.append(i_clamped.unsqueeze(-1) * outer_product)
            
        # Concatenate chunks along sequence dimension
        return torch.cat(result_chunks, dim=1)

    def parallel_gate_computation(self, x_norm):
        """
        Compute all gates in parallel with market regime detection.
        """
        batch_size, seq_len, _ = x_norm.shape
        
        # Detect market regime
        regime_weights = self.regime_detector(x_norm)  # [batch, seq_len, num_regimes]
        
        # Standard gate computation
        q = self.W_q(x_norm)
        k = self.W_k(x_norm) / math.sqrt(self.d)  # Scale keys
        v = self.W_v(x_norm)
        i_tilde = self.W_i(x_norm)
        f_tilde = self.W_f(x_norm)
        
        # Regime-weighted output gate for adaptive behavior in different market conditions
        o_base = self.W_o(x_norm)
        
        # Adapt output gate based on regime - different activations for different regimes
        # This allows model to behave differently in bull/bear/volatile markets
        o = torch.sigmoid(o_base)
        
        # Shape handling
        i_tilde = i_tilde.view(batch_size, seq_len, 1)
        f_tilde = f_tilde.view(batch_size, seq_len, 1)
        
        return q, k, v, i_tilde, f_tilde, o, regime_weights

    def apply_structured_memory_mixing(self, C_t, regime_weights, volatility_scale):
        """
        Apply structured state-space memory mixing with regime awareness,
        using a memory-efficient approach.
        """
        batch_size, seq_len, d, _ = C_t.shape
        
        # Process the memory matrix in chunks to save memory
        chunk_size = 32  # Process 32 rows at a time
        num_chunks = (d + chunk_size - 1) // chunk_size
        
        C_mixed = torch.zeros_like(C_t)
        
        for chunk_idx in range(num_chunks):
            start_idx = chunk_idx * chunk_size
            end_idx = min(start_idx + chunk_size, d)
            
            # Process this chunk of rows
            C_chunk = C_t[:, :, start_idx:end_idx, :]
            
            # Apply SSM processing to each row in the chunk
            for i in range(start_idx, end_idx):
                idx = i - start_idx
                C_row = C_t[:, :, i, :]
                C_row_mixed = self.ssm(C_row, volatility_scale)
                C_mixed[:, :, i, :] = C_row_mixed
        
        # Final memory is adaptively mixed using regime weights
        mixing_factor = regime_weights[:, :, 0:1].unsqueeze(-1)
        
        return C_mixed * mixing_factor + C_t * (1 - mixing_factor)

    def parallel_memory_update(self, v, k, i, f, C_prev, seq_len, regime_weights, volatility_scale):
        """
        Update memory with structured state-space mixing and regime awareness.
        Memory-efficient implementation with chunking.
        """
        batch_size = v.shape[0]
        d = self.d  # Get matrix dimension
        
        # Handle extra dimensions for f once
        if f.dim() > 3:
            f = f.squeeze(-1)
        
        # Ensure i, k, f, v all have the same sequence length
        min_seq_len = min(v.shape[1], k.shape[1], i.shape[1], f.shape[1])
        v = v[:, :min_seq_len]
        k = k[:, :min_seq_len]
        i = i[:, :min_seq_len]
        f = f[:, :min_seq_len]
        regime_weights = regime_weights[:, :min_seq_len]
        volatility_scale = volatility_scale[:, :min_seq_len]
        
        # Update actual sequence length to the trimmed length
        seq_len = min_seq_len
        
        # Calculate cumulative forget product
        f_cum = torch.cumprod(f, dim=1).unsqueeze(-1).unsqueeze(-1)  # [batch, seq, 1, 1]
        
        # Check shape of f_cum, should be [batch, seq_len, 1, 1]
        # If there's an extra dimension, squeeze it
        if f_cum.dim() > 4:
            f_cum = f_cum.squeeze(-1)  # Remove the extra dimension
        
        # Compute key-value outer products
        update_matrices = self.store_key_value(k, v, i)  # [batch, seq, d, d]
        
        # Ensure update_matrices has correct sequence length
        if update_matrices.shape[1] != min_seq_len:
            update_matrices = update_matrices[:, :min_seq_len]
        
        # Compute cumulative updates efficiently
        C_updates = torch.cumsum(update_matrices, dim=1)  # [batch, seq, d, d]
        
        # Expand C_prev for broadcasting - CRITICAL FIX: only expand to min_seq_len
        C_prev_expanded = C_prev.unsqueeze(1).expand(-1, min_seq_len, -1, -1)
        
        # Debug shapes
        # print(f"C_prev_expanded shape: {C_prev_expanded.shape}, f_cum shape: {f_cum.shape}")
        
        # Before expanding, ensure f_cum has compatible dimensions with C_prev_expanded
        f_cum_expanded = f_cum.expand(batch_size, min_seq_len, 1, 1)  # Explicitly define expansion
        
        # Apply forget gates to previous memory and add updates
        C_t = f_cum_expanded * C_prev_expanded + C_updates
        
        # Apply structured mixing and normalization
        C_t = self.apply_structured_memory_mixing(C_t, regime_weights, volatility_scale)
        
        # Final normalization for stability
        C_t = C_t / (torch.norm(C_t, dim=(-2, -1), keepdim=True) + 1e-6) * math.sqrt(self.d)
        
        return C_t

    def parallel_forward(self, x, state):
        """
        Forward pass with parallel sequence processing and financial enhancements.
        """
        batch_size, seq_len, _ = x.size()
        C_prev, n_prev, m_prev = state
        
        # Extract volatility signal for adaptive processing
        volatility_scale = self.detect_volatility(x)
        
        # Input normalization
        x_norm = self.layer_norm_input(x)
        
        # Compute all gates and market regime in parallel
        q, k, v, i_tilde, f_tilde, o, regime_weights = self.parallel_gate_computation(x_norm)
        
        # Stabilized gate computation
        i, f, m_t = self.stabilize_gates(i_tilde, f_tilde, m_prev)
        
        # Shape enforcement
        i = i.view(batch_size, seq_len, 1)
        f = f.view(batch_size, seq_len, 1)
        
        # Memory update with structured state-space mixing
        C_t = self.parallel_memory_update(v, k, i, f, C_prev, seq_len, regime_weights, volatility_scale)
        
        # Get actual sequence length from C_t for consistency
        actual_seq_len = C_t.shape[1]
        
        # Key tracking update (for normalizing query matching)
        n_prev_expanded = n_prev.unsqueeze(1).expand(-1, actual_seq_len, -1)
        
        # Ensure all tensors have the same sequence length as C_t
        q_actual = q[:, :actual_seq_len]
        k_actual = k[:, :actual_seq_len]
        i_actual = i[:, :actual_seq_len]
        f_actual = f[:, :actual_seq_len]
        o_actual = o[:, :actual_seq_len]
        
        i_k_product = i_actual * k_actual
        f_cum = torch.cumprod(f_actual, dim=1)
        n_t = torch.cumsum(i_k_product, dim=1) + f_cum * n_prev_expanded
        
        # Query matching with numerical stability
        h_tilde = torch.einsum('bsde,bse->bsd', C_t, q_actual)
        q_n_dot = torch.sum(n_t * q_actual, dim=-1)
        
        # Stable denominator with market-specific threshold
        denominator = torch.maximum(
            torch.abs(q_n_dot),
            torch.tensor(self.stabilizer_threshold, device=x.device)
        ).unsqueeze(-1)
        
        # Normalized query matching results
        h_tilde = h_tilde / denominator
        
        # Output computation with regime-aware gating
        h_t = o_actual * h_tilde
        
        # Final states for next step
        final_state = (C_t[:, -1], n_t[:, -1], m_t[:, -1])
        
        return h_t, final_state


class ParallelSMLSTM(nn.Module):
    """
    Multi-layer Parallel S-mLSTM optimized for financial time series.
    """
    def __init__(self, input_size, hidden_size, matrix_size=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.matrix_size = matrix_size
        self.num_layers = num_layers
        self.dropout = dropout
        
        # Stack of S-mLSTM cells
        self.cells = nn.ModuleList([
            ParallelSMLSTMCell(
                input_size if i == 0 else hidden_size, 
                hidden_size,
                matrix_size=matrix_size
            )
            for i in range(num_layers)
        ])
        
        self.dropout_layer = nn.Dropout(dropout)
        
        # Financial-specific output adaptation
        self.trend_detector = nn.Linear(hidden_size, 3)  # Bullish, bearish, sideways
        
    def init_state(self, batch_size, device):
        """Initialize hidden states with financial-specific priors."""
        states = []
        for cell in self.cells:
            # Initialize memory matrix with small random values (financial time series benefit from some memory bias)
            C = torch.randn(batch_size, cell.d, cell.d, device=device) * 0.01
            # No initial keys tracked
            n = torch.zeros(batch_size, cell.d, device=device)
            # Initial log-scale at zero
            m = torch.zeros(batch_size, 1, device=device)
            states.append((C, n, m))
        return states
    
    def forward(self, x):
        """Forward pass with checkpointing for memory efficiency."""
        batch_size, seq_len, _ = x.size()
        device = x.device
        
        # Initialize states
        states = self.init_state(batch_size, device)
        
        # Process through layers with residual connections
        layer_input = x
        outputs = []
        
        for layer_idx, cell in enumerate(self.cells):
            # Use checkpointing for memory efficiency with long sequences
            if seq_len > 100 and self.training:
                # Custom checkpointing function to handle state
                def run_cell(inp, state):
                    return cell.parallel_forward(inp, state)
                
                layer_output, states[layer_idx] = checkpoint(
                    run_cell, layer_input, states[layer_idx]
                )
            else:
                layer_output, states[layer_idx] = cell.parallel_forward(
                    layer_input, states[layer_idx]
                )
            
            outputs.append(layer_output)
            
            # Residual connection with dimension matching
            if layer_idx > 0 and layer_input.shape[-1] == layer_output.shape[-1]:
                layer_output = layer_output + layer_input
                
            # Apply dropout between layers
            if layer_idx < self.num_layers - 1:
                layer_output = self.dropout_layer(layer_output)
                
            layer_input = layer_output
        
        # Detect overall market trend from final layer
        trend_logits = self.trend_detector(layer_output.mean(dim=1))
        
        return layer_output, trend_logits


class ParallelMLSTMBlock(nn.Module):
    def __init__(self, input_dim, expanded_dim=None, num_layers=2, dropout=0.2, 
                 expansion_factor=4, lstm_class=ParallelSMLSTM, matrix_size=4):
        super().__init__()
        expanded_dim = expanded_dim or input_dim * expansion_factor
        self.layer_norm = nn.LayerNorm(input_dim)
        self.up_proj = nn.Sequential(
            nn.Linear(input_dim, expanded_dim),
            nn.GELU()
        )
        self.mlstm = lstm_class(
            expanded_dim, 
            expanded_dim // 2,
            matrix_size=matrix_size,
            num_layers=num_layers, 
            dropout=dropout
        )
        self.down_proj = nn.Linear(expanded_dim // 2, input_dim)  # Corrected
        self.skip_scale = nn.Parameter(torch.ones(1))

    def forward(self, x):
        """Forward pass with financial optimization."""
        # Apply normalization for numerical stability
        x_norm = self.layer_norm(x)
        
        # Dimension expansion
        x_up = self.up_proj(x_norm)
        
        # Apply structured mLSTM
        x_mlstm, trend_logits = self.mlstm(x_up)
        
        # Project back to input dimension
        x_down = self.down_proj(x_mlstm)
        
        # Ensure dimensions match for residual connection
        if x.shape != x_down.shape:
            x_down = x_down[:, :x.shape[1], :x.shape[2]]
        
        # Scaled residual connection
        skip_strength = torch.sigmoid(self.skip_scale)
        return x + skip_strength * x_down, trend_logits


class ParallelExtendedSMLSTM(nn.Module):
    """
    Complete financial time series model with structured memory mixing.
    Optimized for stock price prediction with multi-scale modeling.
    """
    def __init__(self, input_size, hidden_size=16, matrix_size=4, num_layers=2, 
                 dropout=0.2, expansion_factor=4):
        super().__init__()
        self.pre_fc_dim = hidden_size * matrix_size * matrix_size
        
        # Financial feature extraction
        self.input_projection = nn.Linear(input_size, self.pre_fc_dim)
        
        # Adaptive normalization (better for varying financial regimes)
        self.layer_norm = nn.LayerNorm(self.pre_fc_dim)
        
        # Multi-scale processing block with financial optimizations
        self.mlstm_block = ParallelMLSTMBlock(
            input_dim=self.pre_fc_dim,
            expanded_dim=self.pre_fc_dim * expansion_factor,
            num_layers=num_layers,
            dropout=dropout,
            expansion_factor=expansion_factor,
            lstm_class=ParallelSMLSTM,
            matrix_size=matrix_size
        )
        
        # Regularization
        self.dropout = nn.Dropout(dropout)
        
        # Final projection with financial-specific initialization
        self.output_projection = nn.Linear(self.pre_fc_dim, self.pre_fc_dim)
        # Initialize with small weights (financial predictions should be conservative)
        self.output_projection.weight.data *= 0.1
        
        # Market regime detector for final output conditioning
        self.regime_detector = nn.Linear(self.pre_fc_dim, 4)
        
        # Trading signal generators (additional outputs beyond price prediction)
        self.signal_generator = nn.Linear(self.pre_fc_dim, 3)  # Buy, hold, sell
    
    def forward(self, x):
        """
        Forward pass with financial-specific processing.
        
        Args:
            x: Input tensor [batch, seq, features]
            
        Returns:
            outputs: Price predictions
            regime: Market regime probabilities
            signals: Trading signals
        """
        # Project input features
        x = self.input_projection(x)
        
        # Normalize (financial data can have extreme values)
        x = self.layer_norm(x)
        
        # Apply structured memory LSTM block
        x, trend_logits = self.mlstm_block(x)
        
        # Nonlinear activation
        x = F.gelu(x)
        
        # Regularization
        x = self.dropout(x)
        
        # Final projection
        outputs = self.output_projection(x)
        
        # Detect market regime
        regime = F.softmax(self.regime_detector(outputs.mean(dim=1)), dim=-1)
        
        # Generate trading signals
        signals = F.softmax(self.signal_generator(outputs.mean(dim=1)), dim=-1)
        
        return outputs, regime, signals

In [ ]:
import torch
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from torch.utils.data import Dataset, DataLoader
import ta
from torch.optim import lr_scheduler, AdamW
import concurrent.futures
import time
from yfinance import download
from requests.exceptions import RequestException
import torch.nn as nn
import torch.nn.functional as F

# Memory optimization settings
torch.cuda.empty_cache()
# Set this to enable memory-efficient operations
torch.backends.cudnn.benchmark = True

# Configurations - REDUCED DIMENSIONS
SEQ_LENGTH = 64  # Reduced from 72
BATCH_SIZE = 32  # Reduced from 64
EPOCHS = 35
FEATURES = ['Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close']



def safe_fetch(ticker, start, end, retries=15, delay=10):
    """
    Attempt to download data with retries and exponential backoff.
    If YFRateLimitError is encountered, wait even longer.
    """
    for attempt in range(retries):
        try:
            data = download(
                ticker,
                start=start.strftime("%Y-%m-%d"), 
                end=end.strftime("%Y-%m-%d"),
                interval='1h',
                auto_adjust=True
            )
            if data is not None and not data.empty:
                return data
        except Exception as yfle:
            print(f"YFRateLimitError on attempt {attempt+1}: {yfle}")
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
        # Exponential backoff
        sleep_time = delay * (attempt + 1)
        print(f"Sleeping for {sleep_time} seconds before next attempt...")
        time.sleep(sleep_time)
    return None


def fetch_chunk(ticker, start, end):
    """
    Fetch a single chunk of data using safe_fetch.
    Adds an extra delay between chunks.
    """
    print(f"Fetching data from {start.date()} to {end.date()}")
    chunk = safe_fetch(ticker, start, end, retries=5, delay=10)
    # Extra delay to reduce request frequency
    time.sleep(5)
    if chunk is not None and not chunk.empty:
        try:
            # Remove multi-index if present
            chunk.columns = chunk.columns.droplevel(1)
        except Exception:
            pass
        return chunk
    else:
        print(f"Warning: no data from {start.date()} to {end.date()}")
        return None

def get_stock_data(ticker, start_date, end_date, max_workers=3):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    delta = pd.Timedelta(days=8)
    
    date_ranges = []
    current_start = start_date
    while (current_start < end_date):
        current_end = min(current_start + delta, end_date)
        date_ranges.append((current_start, current_end))
        current_start = current_end

    data_list = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(fetch_chunk, ticker, s, e) for s, e in date_ranges]
        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            if result is not None and not result.empty:
                data_list.append(result)
                
    if data_list:
        try:
            data = pd.concat(data_list)
            data.sort_index(inplace=True)
            return data
        except ValueError as ve:
            print("No objects to concatenate:", ve)
            return pd.DataFrame()
    else:
        print("No data was fetched!")
        return pd.DataFrame()
    
def get_daily_data(ticker, start_date='2010-01-01', end_date='2023-12-31'):
    file_path = f"{ticker}_daily_{start_date}_{end_date}.csv"
    try:
        data = pd.read_csv(file_path, index_col=0, parse_dates=True)
        print("Loaded daily data from file:", file_path)
    except FileNotFoundError:
        print("File not found, downloading daily data.")
        data = yf.download(ticker, start=start_date, end=end_date, interval='1d')
        data.to_csv(file_path)
        print("Saved daily data to file:", file_path)
    try:
        data.columns = data.columns.droplevel(1)
    except Exception:
        pass
    data.sort_index(inplace=True)
    return data


def get_hourly_data(ticker, start_date, end_date, max_workers=3):
    file_path = f"{ticker}_hourly_{start_date}_{end_date}.csv"
    try:
        data = pd.read_csv(file_path, index_col=0, parse_dates=True)
        print("Loaded hourly data from file:", file_path)
    except FileNotFoundError:
        print("File not found, downloading hourly data.")
        data = get_stock_data(ticker, pd.to_datetime(start_date), pd.to_datetime(end_date), max_workers)
        data.to_csv(file_path)
        print("Saved hourly data to file:", file_path)
    return data

def merge_datasets(daily_df, hourly_df):
    # Read daily data properly, skipping metadata rows
    if 'Ticker' in daily_df.index:
        # Reset index and skip metadata rows
        daily_df = daily_df.reset_index()
        daily_df = daily_df[daily_df['Price'].str.match(r'\d{4}-\d{2}-\d{2}$', na=False)]
        daily_df = daily_df.set_index('Price')
    
    # Ensure indices are datetime
    daily_df.index = pd.to_datetime(daily_df.index)
    hourly_df.index = pd.to_datetime(hourly_df.index)
    
    # Remove timezone info if present
    if hasattr(daily_df.index, "tz") and daily_df.index.tz is not None:
        daily_df.index = daily_df.index.tz_localize(None)
    if hasattr(hourly_df.index, "tz") and hourly_df.index.tz is not None:
        hourly_df.index = hourly_df.index.tz_localize(None)
    
    # Drop any duplicate indices
    daily_df = daily_df[~daily_df.index.duplicated(keep='first')]
    hourly_df = hourly_df[~hourly_df.index.duplicated(keep='first')]
    
    # Sort both dataframes by index
    daily_df = daily_df.sort_index()
    hourly_df = hourly_df.sort_index()
    
    # Resample daily data to hourly frequency
    daily_up = daily_df.resample('1h').ffill()
    
    # Merge the datasets
    merged = pd.concat([daily_up, hourly_df])
    
    # Handle duplicates keeping the most recent data
    merged = merged[~merged.index.duplicated(keep='last')]
    
    # Final sort
    merged = merged.sort_index()
    
    return merged



def add_technical_indicators(df):
    if len(df) < 20:  
        print("Data is too short, skipping indicators.")
        return df.copy()

    # Convert string columns to numeric
    numeric_columns = ['Close', 'High', 'Low', 'Volume']
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.copy().ffill().bfill()    
    close = df['Close']
    high = df['High']
    low = df['Low']
    volume = df['Volume']

    # Rest of your indicators code...
    rsi_indicator = ta.momentum.RSIIndicator(close=close)
    df['RSI'] = rsi_indicator.rsi()

    macd = ta.trend.MACD(close=close)
    df['MACD'] = macd.macd()
    df['MACD_signal'] = macd.macd_signal()

    bollinger = ta.volatility.BollingerBands(close=close)
    df['BB_upper'] = bollinger.bollinger_hband()
    df['BB_middle'] = bollinger.bollinger_mavg()
    df['BB_lower'] = bollinger.bollinger_lband()

    atr = ta.volatility.AverageTrueRange(high=high, low=low, close=close)
    df['ATR'] = atr.average_true_range()

    df['Returns'] = close.pct_change()
    df['Log_Returns'] = np.log1p(df['Returns'].replace([-np.inf, np.inf], np.nan))
    df['Volatility'] = df['Returns'].rolling(window=20).std()

    df['Volume_MA'] = volume.rolling(window=20).mean()
    df['Volume_STD'] = volume.rolling(window=20).std()

    df.index = pd.to_datetime(df.index)
    df['DayOfWeek'] = df.index.dayofweek
    df['MonthOfYear'] = df.index.month

    df = df.dropna().ffill().bfill().replace([np.inf, -np.inf], np.nan).fillna(df.mean())
    return df


class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).squeeze(-1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]






###############################################################################
# xLSTM
###############################################################################



class FunnyMachine(nn.Module):
    def __init__(self, input_size, hidden_size=16, matrix_size=2, dropout=0.2):
        super().__init__()
        # Replace with structured memory LSTM with smaller dimensions
        self.extended_mlstm = ParallelExtendedSMLSTM(
            input_size=input_size,
            hidden_size=hidden_size,     # Reduced from 32
            matrix_size=matrix_size,     # Reduced from 4
            num_layers=2,
            dropout=dropout,
            expansion_factor=2,          # Reduced from 4
        )
        self.output_layer = nn.Linear(hidden_size * matrix_size**2, 5)
        
        # New: market regime-aware output adaptation
        self.regime_adapter = nn.Linear(4, 5)  # 4 regime probabilities to 5 outputs

    def forward(self, x):
        # The new model returns multiple outputs
        outputs, regime, signals = self.extended_mlstm(x)
        outputs = F.gelu(outputs)  # Apply GELU activation
        
        # Pool across sequence dimension
        pooled_output = outputs.mean(1)
        
        # Base predictions
        base_predictions = self.output_layer(pooled_output)
        
        # Regime-conditioned adjustment (small influence)
        regime_adjustment = self.regime_adapter(regime) * 0.1
        
        # Combine base predictions with regime awareness
        final_predictions = base_predictions + regime_adjustment
        
        return final_predictions, regime, signals

def prepare_data(df):
    df = df.copy()
    print("Initial data size:", len(df))
    
    # Add more robust handling of NaN and infinity values
    df = df.ffill().bfill()
    df = df.replace([np.inf, -np.inf], np.nan)

    close = df['Close']
    high = df['High']
    low = df['Low']
    volume = df['Volume']
    
    # PRICE DYNAMICS - minimal overlap
    df['Returns'] = close.pct_change()  # Keep daily returns as baseline
    # More careful log1p handling
    df['Returns'] = df['Returns'].replace([np.inf, -np.inf], np.nan)
    df['Log_Returns'] = np.log1p(df['Returns'].clip(-0.99, 10))  # Clip to avoid infinity
    
    # TREND INDICATORS - different timeframes
    df['EMA_Short'] = close.ewm(span=12, adjust=False).mean() / close - 1  # Short-term trend
    df['EMA_Medium'] = close.ewm(span=26, adjust=False).mean() / close - 1  # Medium-term trend
    df['EMA_Long'] = close.ewm(span=50, adjust=False).mean() / close - 1  # Long-term trend
    
    # MOMENTUM INDICATORS - distinct calculations
    df['RSI'] = ta.momentum.RSIIndicator(close=close, window=14).rsi() / 100  # Normalized
    df['MACD'] = ta.trend.MACD(close=close).macd_diff()  # Only difference, more informative
    df['ROC'] = ta.momentum.ROCIndicator(close=close, window=10).roc()  # Rate of change
    
    # VOLATILITY INDICATORS
    df['ATR_Norm'] = ta.volatility.AverageTrueRange(
        high=high, low=low, close=close, window=14
    ).average_true_range() / close  # Normalized by price
    
    bb = ta.volatility.BollingerBands(close=close, window=20)
    df['BB_Width'] = (bb.bollinger_hband() - bb.bollinger_lband()) / bb.bollinger_mavg()  # Width only
    
    # VOLUME INDICATORS - distinct aspects
    df['OBV_Change'] = ta.volume.OnBalanceVolumeIndicator(
        close=close, volume=volume
    ).on_balance_volume().pct_change()  # OBV momentum
    
    df['Volume_Price_Trend'] = ta.volume.VolumePriceTrendIndicator(
        close=close, volume=volume
    ).volume_price_trend()  # Volume-price relationship
    
    # SUPPORT/RESISTANCE INDICATORS
    df['Stochastic_K'] = ta.momentum.StochasticOscillator(
        high=high, low=low, close=close
    ).stoch() / 100  # Position within range
    
    # MARKET REGIME INDICATORS
    df['ADX'] = ta.trend.ADXIndicator(high=high, low=low, close=close).adx() / 100  # Trend strength
    
    # CYCLICAL FEATURES
    df['DayOfWeek'] = df.index.dayofweek / 6  # Normalized 0-1
    df['HourOfDay'] = df.index.hour / 23  # Normalized 0-1
    df['DayOfMonth'] = df.index.day / 31  # Normalized 0-1
    
    # LAGGED FEATURES - important for sequence modeling
    df['Price_Change_Lag1'] = df['Returns'].shift(1)
    
    # CROSS-ASSET FEATURES (if available)
    # df['SPY_Correlation'] = ...  # Would require additional data
    
    print("After technical indicators:", len(df))
    df = df.dropna().ffill().bfill().replace([np.inf, -np.inf], np.nan).fillna(df.mean())
    
    return df

def create_sequences(scaled_data, target_scaled, seq_length, augment=True):
    X, y = [], []
    for i in range(seq_length, len(scaled_data) - 5):
        seq = scaled_data[i - seq_length:i]
        
        if augment:
            
            if np.random.random() < 0.2:
                
                noise = np.random.normal(0, 0.002 * np.std(seq), seq.shape)
                seq = seq + noise
        
            if np.random.random() < 0.15:
                
                scale = np.random.uniform(0.98, 1.02)
                seq = seq * scale
            if np.random.random() < 0.1:  # New: time warping
                warp = np.random.uniform(0.98, 1.02, seq.shape)
                seq = seq * warp

        X.append(seq)
        y.append(target_scaled[i:i+5].flatten())
    return np.array(X), np.array(y)

def train_model(model, train_loader, val_loader, target_scaler, epochs=100):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    model.to(device)
    
    # Enable mixed precision training with updated syntax
    if torch.__version__ >= '2.0.0':
        # New syntax for PyTorch 2.0+
        scaler = torch.amp.GradScaler('cuda')
    else:
        # Legacy syntax for older versions
        scaler = torch.cuda.amp.GradScaler()
    
    criterion = nn.HuberLoss(delta=0.5)
    optimizer = AdamW(
        model.parameters(), 
        lr=2e-4,
        weight_decay=1e-4,
        eps=1e-8
    )
    
    l1_lambda = 1e-5  
    scheduler = lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=5e-4,
        div_factor=10,
        final_div_factor=1e4,
        steps_per_epoch=len(train_loader),
        epochs=epochs,
        pct_start=0.3,
        anneal_strategy='cos'
    )
    
    best_val_loss = float('inf')
    patience = 7  
    min_delta = 1e-4  
    waiting = 0
    GRAD_CLIP_VALUE = 0.5
    
    regime_history = []  # Track market regime predictions
    
    # Memory tracking
    if device.type == 'cuda':
        print(f"Initial GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            
            # Updated mixed precision context manager
            if torch.__version__ >= '2.0.0':
                # New syntax for PyTorch 2.0+
                with torch.amp.autocast('cuda'):
                    outputs, regime, signals = model(X_batch)
                    l1_norm = sum(p.abs().sum() for p in model.parameters())
                    loss = criterion(outputs, y_batch) + l1_lambda * l1_norm
            else:
                # Legacy syntax for older versions
                with torch.cuda.amp.autocast():
                    outputs, regime, signals = model(X_batch)
                    l1_norm = sum(p.abs().sum() for p in model.parameters())
                    loss = criterion(outputs, y_batch) + l1_lambda * l1_norm
            
            # Mixed precision backward pass
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_VALUE)
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            
            # Clear cache to prevent memory accumulation
            if device.type == 'cuda':
                torch.cuda.empty_cache()
        
        model.eval()
        val_loss = 0
        all_outputs = []
        all_targets = []
        all_regimes = []
        all_signals = []
        
        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val, y_val = X_val.to(device), y_val.to(device)
                
                # Updated mixed precision context manager
                if torch.__version__ >= '2.0.0':
                    with torch.amp.autocast('cuda'):
                        outputs, regime, signals = model(X_val)
                        val_loss += criterion(outputs, y_val).item()
                else:
                    with torch.cuda.amp.autocast():
                        outputs, regime, signals = model(X_val)
                        val_loss += criterion(outputs, y_val).item()
                
                # Move results to CPU immediately to free GPU memory
                all_outputs.append(outputs.cpu())
                all_targets.append(y_val.cpu())
                all_regimes.append(regime.cpu())
                all_signals.append(signals.cpu())
                
                # Clear cache after each batch
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
                    
        # Rest of evaluation code with tensors already on CPU
        all_outputs = torch.cat(all_outputs, dim=0)
        all_targets = torch.cat(all_targets, dim=0)
        all_regimes = torch.cat(all_regimes, dim=0)
        all_signals = torch.cat(all_signals, dim=0)
        
        # Calculate metrics
        mape = torch.mean(torch.abs((all_targets - all_outputs) / (all_targets + 1e-8))) * 100
        rmse = torch.sqrt(torch.mean((all_targets - all_outputs) ** 2))
        mae = torch.mean(torch.abs(all_targets - all_outputs))
        
        # Directional accuracy 
        if all_targets.shape[1] > 1:
            diff_pred = all_outputs[:, 1:] - all_outputs[:, :-1]
            diff_true = all_targets[:, 1:] - all_targets[:, :-1]
            directional_accuracy = torch.mean((torch.sign(diff_pred) == torch.sign(diff_true)).float()) * 100
        else:
            directional_accuracy = torch.tensor(0.0)
        
        # Calculate trading signal accuracy (if we had ground truth)
        # Since we don't, just monitor distribution
        signal_distribution = all_signals.mean(dim=0)
        regime_distribution = all_regimes.mean(dim=0)
        
        # Store regime history for later analysis
        regime_history.append(regime_distribution.numpy())
        
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        scheduler.step()
        
        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'MAPE: {mape:.2f}%, RMSE: {rmse:.4f}, MAE: {mae:.4f}, '
              f'Directional Accuracy: {directional_accuracy:.2f}%')
        print(f'Regime Distribution: {regime_distribution.numpy()}')
        print(f'Signal Distribution: {signal_distribution.numpy()}')
        
        if val_loss < (best_val_loss - min_delta):
            best_val_loss = val_loss
            waiting = 0
            print("Saving model...")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'mape': mape,
                'rmse': rmse,
                'mae': mae,
                'directional_accuracy': directional_accuracy,
                'regime_distribution': regime_distribution,
                'signal_distribution': signal_distribution,
                'regime_history': regime_history,
            }, 'smLSTM_predictor_MSFT.pth')  # New filename for new model
        else:
            waiting += 1
            if waiting >= patience:
                print(f'Early stopping triggered at epoch {epoch+1}')
                break

        # Print memory usage
        if device.type == 'cuda':
            print(f"GPU memory after epoch {epoch+1}: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

def analyze_predictions(model, test_loader, target_scaler):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    all_outputs = []
    all_targets = []
    all_regimes = []
    all_signals = []
    
    with torch.no_grad():
        for X_test, y_test in test_loader:
            X_test, y_test = X_test.to(device), y_test.to(device)
            outputs, regime, signals = model(X_test)
            
            all_outputs.append(outputs.cpu())
            all_targets.append(y_test.cpu())
            all_regimes.append(regime.cpu())
            all_signals.append(signals.cpu())
    
    all_outputs = torch.cat(all_outputs, dim=0).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    all_regimes = torch.cat(all_regimes, dim=0).numpy()
    all_signals = torch.cat(all_signals, dim=0).numpy()
    
    # Convert scaled predictions back to original price
    pred_prices = target_scaler.inverse_transform(all_outputs)
    actual_prices = target_scaler.inverse_transform(all_targets)
    
    # Financial-specific metrics
    mape = np.mean(np.abs((actual_prices - pred_prices) / actual_prices)) * 100
    rmse = np.sqrt(np.mean((actual_prices - pred_prices) ** 2))
    
    # Directional accuracy (critical for trading)
    diff_pred = np.diff(pred_prices, axis=1)
    diff_actual = np.diff(actual_prices, axis=1)
    dir_acc = np.mean((np.sign(diff_pred) == np.sign(diff_actual)))
    
    # Analyze by detected market regime
    regime_labels = ["Bull", "Bear", "Sideways", "Volatile"]
    dominant_regimes = np.argmax(all_regimes, axis=1)
    
    for i in range(len(regime_labels)):
        regime_mask = dominant_regimes == i
        if np.sum(regime_mask) > 0:
            regime_mape = np.mean(np.abs((actual_prices[regime_mask] - pred_prices[regime_mask]) / 
                                actual_prices[regime_mask])) * 100
            print(f"{regime_labels[i]} Market Regime - MAPE: {regime_mape:.2f}%, Samples: {np.sum(regime_mask)}")
    
    # Signal analysis
    signal_labels = ["Buy", "Hold", "Sell"]
    recommended_signals = np.argmax(all_signals, axis=1)
    signal_counts = np.bincount(recommended_signals, minlength=3)
    
    print("\nTrading Signal Distribution:")
    for i, label in enumerate(signal_labels):
        print(f"{label}: {signal_counts[i]} ({signal_counts[i]/len(recommended_signals)*100:.1f}%)")
    
    return {
        'mape': mape,
        'rmse': rmse,
        'directional_accuracy': dir_acc,
        'predictions': pred_prices,
        'actuals': actual_prices,
        'regimes': all_regimes,
        'signals': all_signals
    }

def main():
    alreadygot = False
    if (alreadygot):
        daily_data = pd.read_csv('/kaggle/input/msftdata/MSFT_daily_2010-01-01_2023-12-31.csv', 
                        skiprows=[1,2],  # Skip the metadata rows
                        index_col=0,
                        parse_dates=True)
        hourly_data = pd.read_csv('/kaggle/input/msftdata/MSFT_hourly_2023-02-24_2025-02-16.csv', 
                         index_col=0, 
                         parse_dates=True)
    else: 
        daily_data = pd.read_csv('/kaggle/input/msftdata/MSFT_daily_2010-01-01_2023-12-31.csv', 
                        skiprows=[1,2],  # Skip the metadata rows
                        index_col=0,
                        parse_dates=True)
        hourly_data = pd.read_csv('/kaggle/input/msftdata/MSFT_hourly_2023-02-24_2025-02-16.csv', 
                         index_col=0, 
                         parse_dates=True)

        # Convert data types after loading
        numeric_columns = ['Close', 'High', 'Low', 'Open', 'Volume']
        for col in numeric_columns:
            daily_data[col] = pd.to_numeric(daily_data[col], errors='coerce')
            hourly_data[col] = pd.to_numeric(hourly_data[col], errors='coerce')

        df = merge_datasets(daily_data, hourly_data)

    df = add_technical_indicators(df)
    processed_df = prepare_data(df)
    print("After prepare_data:", len(processed_df))

    target = processed_df['Close'].values.reshape(-1, 1)
    features = processed_df.drop(columns=['Close'])
    tscv = TimeSeriesSplit(n_splits=5)

    for fold, (train_idx, val_idx) in enumerate(tscv.split(features)):
        print(f"Training fold {fold+1}")
        X_train, X_val = features.iloc[train_idx], features.iloc[val_idx]
        y_train, y_val = target[train_idx], target[val_idx]

        feature_scaler = RobustScaler()
        target_scaler = RobustScaler()
        X_train_scaled = feature_scaler.fit_transform(X_train)
        X_val_scaled = feature_scaler.transform(X_val)
        y_train_scaled = target_scaler.fit_transform(y_train)
        y_val_scaled = target_scaler.transform(y_val)

        X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, SEQ_LENGTH)
        X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val_scaled, SEQ_LENGTH)
        print("Sequence size:", len(X_train_seq))
        print(f"Train sequences: {X_train_seq.shape}, Targets: {y_train_seq.shape}")

        # Use smaller subset of data if memory is constrained
        max_samples = 5000  # Limit the number of samples for memory
        X_train_seq = X_train_seq[:max_samples]
        y_train_seq = y_train_seq[:max_samples]
        X_val_seq = X_val_seq[:min(1000, len(X_val_seq))]  # Also limit validation samples
        y_val_seq = y_val_seq[:min(1000, len(y_val_seq))]
        
        print(f"Using reduced dataset - Train: {len(X_train_seq)}, Val: {len(X_val_seq)}")
        
        train_dataset = StockDataset(X_train_seq, y_train_seq)
        val_dataset = StockDataset(X_val_seq, y_val_seq)
        
        # Use num_workers=0 to avoid extra memory usage with multiprocessing
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)
        
        model = FunnyMachine(
            input_size=X_train_seq.shape[2],
            hidden_size=16,    # Reduced from 32
            matrix_size=4,     # Reduced from 4
            dropout=0.2
        )
        
        print("Enhanced s_mLSTM Stock Prediction Model:")
        print(model)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total_params:,}")
        
        # Create feature importance tracker to monitor which financial indicators matter most
        feature_names = list(features.columns)
        print(f"Using {len(feature_names)} features: {feature_names}")
        
        train_model(model, train_loader, val_loader, target_scaler, epochs=EPOCHS)
        break

if __name__ == "__main__":
    main()


/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Initial data size: 123879


/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/c

After technical indicators: 123879
After prepare_data: 123846
Training fold 1
Sequence size: 20572
Train sequences: (20572, 64, 31), Targets: (20572, 5)
Using reduced dataset - Train: 5000, Val: 1000
Enhanced s_mLSTM Stock Prediction Model:
FunnyMachine(
  (extended_mlstm): ParallelExtendedSMLSTM(
    (input_projection): Linear(in_features=31, out_features=256, bias=True)
    (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (mlstm_block): ParallelMLSTMBlock(
      (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (up_proj): Sequential(
        (0): Linear(in_features=256, out_features=512, bias=True)
        (1): GELU(approximate='none')
      )
      (mlstm): ParallelSMLSTM(
        (cells): ModuleList(
          (0): ParallelSMLSTMCell(
            (W_q): Linear(in_features=512, out_features=256, bias=True)
            (W_k): Linear(in_features=512, out_features=256, bias=True)
            (W_v): Linear(in_features=512, out_features=256,